# Chemical Mechanism Classification

**Arquitectura híbrida:**
1. **Parser determinístico** — resuelve ~89% de los casos con 94.3% accuracy
   - Semántica inline con fallback de dos pasadas para casos con 0 sobrevivientes
2. **Ensemble ML (LightGBM + CatBoost)** — tiebreaker para casos ambiguos (75.7% OOF)
3. **Claude API** — fallback opcional, se detecta automáticamente

**OOF combinado: 92.2%** | Baseline AI: 77.4% | Leaderboard #1: 95.8%

## 1. Setup e importaciones

In [13]:
import subprocess, sys
def pip(pkg): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
pip('lightgbm')
pip('xgboost')
pip('scikit-learn')
pip('catboost')

In [1]:
import pandas as pd
import numpy as np
import re
import json
import time
import urllib.request
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

MECHS = ['Alpha transformation', 'Beta displacement',
         'Gamma rearrangement', 'Delta elimination']

print('Libraries loaded OK')

Libraries loaded OK


## 2. Carga de datos

In [11]:
PLATFORM_DATA_DIR = Path('dataset/public')

if PLATFORM_DATA_DIR.exists():
    TRAIN_PATH  = PLATFORM_DATA_DIR / 'train.csv'
    TEST_PATH   = PLATFORM_DATA_DIR / 'test.csv'
    SUBMIT_PATH = Path('working/submission.csv')
    Path('working').mkdir(exist_ok=True)
    print(f'Plataforma detectada: {PLATFORM_DATA_DIR}')
else:
    from google.colab import files
    print('Subí train.csv y test.csv:')
    files.upload()
    TRAIN_PATH  = Path('train.csv')
    TEST_PATH   = Path('test.csv')
    SUBMIT_PATH = Path('submission.csv')

train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)
print(f'Train: {len(train_df):,} filas | Test: {len(test_df):,} filas')

Subí train.csv y test.csv:


Saving train.csv to train (1).csv
Saving test.csv to test (1).csv
Saving sample_submission.csv to sample_submission (1).csv
Train: 7,000 filas | Test: 3,000 filas


## 3. Parser determinístico

**Semánticas descubiertas empíricamente:**
- Todos los mecanismos empiezan activos
- `blocked` gana sobre `activated`: una vez bloqueado no se puede reactivar
- `override` es forward-only: cancela reglas que vienen DESPUÉS en la secuencia
- `dominance_reversed` sin dominancia previa → crea (B domina A)
- Dominancia se aplica inline al disparar la regla
- `exactly_N` se aplica iterativamente al final hasta estabilizar
- **Fallback**: si el inline deja 0 activos, reintenta con dominancia aplicada
  solo sobre mecanismos que sobreviven al conjunto final de bloqueados

In [23]:
# ── Helpers de parsing ───────────────────────────────────────────────────

def parse_state(state_str: str) -> dict:
    result = {}
    for part in state_str.split(', '):
        k, v = part.split('=')
        result[k.strip()] = float(v.strip())
    return result


def eval_condition(cond_str: str, state: dict) -> bool:
    expr = cond_str
    for var, val in sorted(state.items(), key=lambda x: -len(x[0])):
        expr = expr.replace(var, str(val))
    expr = (expr
            .replace(' AND ', ' and ')
            .replace(' OR ',  ' or ')
            .replace('NOT ',  'not '))
    try:
        return bool(eval(expr))
    except Exception:
        return False


def parse_rule(rule_text: str):
    m = re.match(r'If exactly (\d+) mechanisms remain, eliminate (.+?)\.?$', rule_text)
    if m: return {'type':'exactly_N','N':int(m.group(1)),'target':m.group(2).strip()}
    m = re.match(r'If (.+), Rule (\d+) is overridden\.?$', rule_text)
    if m: return {'type':'override','condition':m.group(1),'target_rule':int(m.group(2))}
    m = re.match(r'If (.+), (.+) is (activated|blocked)\.?$', rule_text)
    if m: return {'type':m.group(3),'condition':m.group(1),'target':m.group(2).strip()}
    m = re.match(r'If (.+), dominance between (.+) and (.+) is reversed\.?$', rule_text)
    if m: return {'type':'dominance_reversed','condition':m.group(1),
                  'mech_a':m.group(2).strip(),'mech_b':m.group(3).strip()}
    m = re.match(r'If (.+), (.+) dominates (.+?)\.?$', rule_text)
    if m: return {'type':'dominates','condition':m.group(1),
                  'dominator':m.group(2).strip(),'dominated':m.group(3).strip()}
    return None


def parse_all_rules(rules_text: str):
    rules = []
    for line in rules_text.strip().split('\n'):
        mm = re.match(r'^(\d+)\.\s+(.+)$', line.strip())
        if mm:
            p = parse_rule(mm.group(2))
            if p: rules.append((int(mm.group(1)), p))
    rules.sort(key=lambda x: x[0])
    return rules


def apply_dominance(active: set, dominance: set):
    changed = True
    while changed:
        changed = False
        for (dom, dom_ed) in list(dominance):
            if dom in active and dom_ed in active:
                active.discard(dom_ed); changed = True


print('Helpers definidos')

Helpers definidos


In [38]:
# ── Solver principal ──────────────────────────────────────────────────────
def apply_exactly_n(active, rules, overridden):
    """Aplica reglas exactly_N iterativamente hasta estabilizar."""
    exactly_n = [(i,r) for i,r in rules
                 if r['type'] == 'exactly_N' and i not in overridden]
    changed = True
    while changed:
        changed = False
        for i, r in exactly_n:
            if len(active) == r['N'] and r['target'] in active:
                active.discard(r['target']); changed = True

# ── Solver v2: inline dominance (semántica original) ─────────────────────
def _solve_inline(rules, state, overridden, pm):
    active = set(MECHS); blocked_set = set(); dominance = set()
    for idx, rule in rules:
        is_ov = idx in overridden; t = rule['type']
        if t in ('blocked', 'activated'):
            tgt = rule.get('target', '')
            if tgt not in pm: continue
            cv = eval_condition(rule['condition'], state) if not is_ov else False
            bi = 0 if t == 'blocked' else 1
            if is_ov:   pm[tgt][11 + bi] += 1
            elif cv:    pm[tgt][bi] += 1
            else:       pm[tgt][7 + bi] += 1
            if not is_ov and cv:
                if t == 'blocked':
                    blocked_set.add(tgt); active.discard(tgt)
                elif tgt not in blocked_set:
                    active.add(tgt)
        elif t == 'dominates':
            dom, domed = rule.get('dominator',''), rule.get('dominated','')
            cv = eval_condition(rule['condition'], state) if not is_ov else False
            if dom in pm:
                if is_ov: pm[dom][13] += 1
                elif cv:  pm[dom][2] += 1; dominance.add((dom, domed)); apply_dominance(active, dominance)
                else:     pm[dom][9] += 1
            if domed in pm:
                if is_ov: pm[domed][14] += 1
                elif cv:  pm[domed][3] += 1
                else:     pm[domed][10] += 1
        elif t == 'dominance_reversed':
            a, b = rule.get('mech_a',''), rule.get('mech_b','')
            cv = eval_condition(rule['condition'], state) if not is_ov else False
            if cv and not is_ov:
                if a in pm: pm[a][4] += 1
                if b in pm: pm[b][5] += 1
                if   (a,b) in dominance: dominance.discard((a,b)); dominance.add((b,a))
                elif (b,a) in dominance: dominance.discard((b,a)); dominance.add((a,b))
                else:                    dominance.add((b,a))
                apply_dominance(active, dominance)
        elif t == 'exactly_N':
            tgt = rule.get('target', '')
            if tgt in pm and not is_ov:
                pm[tgt][6]  += 1
                pm[tgt][15] += len(active)
                pm[tgt][16] += (1 if len(active) == rule['N'] else 0)
    apply_dominance(active, dominance)
    apply_exactly_n(active, rules, overridden)
    return active


# ── Solver v3: post-block dominance ──────────────────────────────────────
# Primero resuelve todos los blocked/activated; dominancia solo aplica
# si el dominador sobrevive al conjunto final de bloqueos.
def _solve_postblock(rules, state, ov):
    """
    Modo F: domrev puede revertir dominates + crear si vacío.
            domrev NO puede revertir otro domrev.
    """
    fb = set(); apb = set(MECHS)
    for idx, rule in rules:
        if idx in ov or rule['type'] not in ('blocked', 'activated'):
            continue
        if not eval_condition(rule['condition'], state):
            continue
        if rule['type'] == 'blocked':
            fb.add(rule['target']); apb.discard(rule['target'])
        elif rule['target'] not in fb:
            apb.add(rule['target'])

    # dom guarda (a,b) -> 'dom' | 'domrev' para distinguir origen
    dom = {}
    for idx, rule in rules:
        if idx in ov:
            continue
        t = rule['type']
        if t == 'dominates':
            d, de = rule['dominator'], rule['dominated']
            if eval_condition(rule['condition'], state) and d not in fb:
                dom[(d, de)] = 'dom'
        elif t == 'dominance_reversed':
            a, b = rule['mech_a'], rule['mech_b']
            if not eval_condition(rule['condition'], state):
                continue
            prev_ab = dom.get((a, b))
            prev_ba = dom.get((b, a))
            if prev_ab == 'dom':
                del dom[(a, b)]
                if b not in fb:   dom[(b, a)] = 'domrev'
                elif a not in fb: dom[(a, b)] = 'domrev'
            elif prev_ba == 'dom':
                del dom[(b, a)]
                if a not in fb:   dom[(a, b)] = 'domrev'
                elif b not in fb: dom[(b, a)] = 'domrev'
            elif not prev_ab and not prev_ba:
                # sin previa → crear
                if b not in fb:   dom[(b, a)] = 'domrev'
                elif a not in fb: dom[(a, b)] = 'domrev'
            # si prev es 'domrev' → no hace nada

    dom_set = set(dom.keys())
    active = set(apb)
    apply_dominance(active, dom_set)
    apply_exactly_n(active, rules, ov)
    return active


def _get_final_blocked(rules, state, ov):
    """Calcula el conjunto final de mecanismos bloqueados."""
    fb = set()
    for idx, rule in rules:
        if idx in ov or rule['type'] not in ('blocked', 'activated'): continue
        if not eval_condition(rule['condition'], state): continue
        if rule['type'] == 'blocked':  fb.add(rule['target'])
        elif rule['target'] in fb:     fb.discard(rule['target'])
    return fb


def solve_both(rules_text, state_str):
    """
    Corre v2 (inline) y v3 (post-block).
    Selecciona el solver primario según `has_blocked_dominator`:
      - Si hay dominador bloqueado → v3 es más confiable
      - Si no                      → v2 es más confiable
    Retorna: (active_primary, active_v2, active_v3, has_bd, pm, state_vec, global_feats)
    """
    state = parse_state(state_str)
    rules = parse_all_rules(rules_text)
    pm = {m: np.zeros(17) for m in MECHS}

    overridden = set()
    for idx, rule in rules:
        if rule['type'] == 'override' and eval_condition(rule['condition'], state):
            overridden.add(rule['target_rule'])

    # Detectar si algún dominador activo termina bloqueado
    fb = _get_final_blocked(rules, state, overridden)
    has_bd = False
    for idx, rule in rules:
        if idx in overridden: continue
        t = rule['type']
        if t == 'dominates':
            if eval_condition(rule['condition'], state) and rule['dominator'] in fb:
                has_bd = True; break
        elif t == 'dominance_reversed':
            if eval_condition(rule['condition'], state):
                if rule.get('mech_a','') in fb or rule.get('mech_b','') in fb:
                    has_bd = True; break

    active_v2 = _solve_inline(rules, state, overridden, pm)
    if len(active_v2) == 0:
        active_v2 = _solve_postblock(rules, state, overridden)

    active_v3 = _solve_postblock(rules, state, overridden)

    # Selección del solver primario
    active_primary = active_v3 if has_bd else active_v2
    if len(active_primary) == 0:
        active_primary = active_v2 if has_bd else active_v3

    sv   = np.array([state[v] for v in
                     ['sigma_density','steric_factor','polarity','thermal_index','zeta_affinity']])
    n_en = sum(1 for i, r in rules if r['type'] == 'exactly_N' and i not in overridden)
    gf   = np.array([len(rules), len(overridden), n_en, 0])
    return active_primary, active_v2, active_v3, has_bd, pm, sv, gf


# ── Validación rápida ──────────────────────────────────────────────────
det_c = det_t = 0
for _, row in train_df.iterrows():
    opts = {'A':row['option_A'],'B':row['option_B'],'C':row['option_C'],'D':row['option_D']}
    cm   = opts[row['answer']]
    ap, a2, a3, has_bd, _, _, _ = solve_both(row['rules_text'], row['reaction_state'])
    if len(ap) == 1:
        det_t += 1
        if list(ap)[0] == cm: det_c += 1

print(f'Solver primario det: {det_t:,}  acc={det_c/det_t:.4f}')
print(f'has_blocked_dom logic OK')



Solver primario det: 6,169  acc=0.9674
has_blocked_dom logic OK


## 4. Ensemble ML tiebreaker

In [39]:
try:
    import lightgbm as lgb
    from catboost import CatBoostClassifier
    ML_AVAILABLE = True
    print('LightGBM y CatBoost disponibles ✓')
except ImportError as e:
    ML_AVAILABLE = False
    print(f'ML no disponible ({e}) — usando heurística de dominancia para tiebreak')

LightGBM y CatBoost disponibles ✓


In [40]:
def build_feature_vector(pm, state_vec, global_feats, mech_idx,
                         in_v2, in_v3, in_primary, agree,
                         na2, na3, n_primary,
                         in_inter, n_inter, has_bd):
    return np.concatenate([
        pm[MECHS[mech_idx]],   #  17
        state_vec,             #  5
        global_feats,          #  4
        [in_v2,                #  1
         in_v3,                #  1
         in_primary,           #  1  ← nuevo: solver elegido por has_bd
         agree,                #  1
         na2, na3, n_primary,  #  3  ← n_primary nuevo
         in_inter, n_inter,    #  2
         has_bd,               #  1  ← KEY: flag de dominador bloqueado
         mech_idx]             #  1
    ])  # total: 35 features


def build_dataset(df):
    X_rows, y_rows, prob_ids = [], [], []
    for _, row in df.iterrows():
        opts = {'A':row['option_A'],'B':row['option_B'],'C':row['option_C'],'D':row['option_D']}
        cm   = opts[row['answer']]
        ap, a2, a3, has_bd, pm, sv, gf = solve_both(row['rules_text'], row['reaction_state'])
        na2, na3, nap = len(a2), len(a3), len(ap)
        agr   = 1 if (na2==1 and na3==1 and a2==a3) else 0
        inter = a2 & a3;  n_inter = len(inter)
        for i, m in enumerate(MECHS):
            feat = build_feature_vector(
                pm, sv, gf, i,
                1 if m in a2 else 0,
                1 if m in a3 else 0,
                1 if m in ap else 0,
                agr, na2, na3, nap,
                1 if m in inter else 0, n_inter,
                1 if has_bd else 0)
            X_rows.append(feat)
            y_rows.append(1 if m == cm else 0)
            prob_ids.append(row['problem_id'])
    return np.array(X_rows), np.array(y_rows), np.array(prob_ids)


clf_lgb = clf_cat = None

if ML_AVAILABLE:
    clf_lgb = lgb.LGBMClassifier(
        n_estimators=300, num_leaves=63, learning_rate=0.05,
        n_jobs=-1, random_state=42, verbose=-1)
    clf_cat = CatBoostClassifier(
        iterations=300, depth=6, learning_rate=0.05,
        random_seed=42, verbose=0)

print('Modelos instanciados')

Modelos instanciados


In [41]:
if ML_AVAILABLE:
    print('Construyendo dataset...')
    X, y, problem_ids = build_dataset(train_df)
    print(f'Shape: {X.shape} | Positivos: {y.mean():.3f}')

    unique_probs = np.unique(problem_ids)
    np.random.seed(42); np.random.shuffle(unique_probs)
    folds = np.array_split(unique_probs, 5)

    oof_lgb = np.zeros(len(X))
    oof_cat = np.zeros(len(X))

    print('5-fold CV...')
    for k, val_probs in enumerate(folds):
        tm = ~np.isin(problem_ids, val_probs)
        vm =  np.isin(problem_ids, val_probs)
        clf_lgb.fit(X[tm], y[tm])
        clf_cat.fit(X[tm], y[tm])
        oof_lgb[vm] = clf_lgb.predict_proba(X[vm])[:, 1]
        oof_cat[vm] = clf_cat.predict_proba(X[vm])[:, 1]
        print(f'  Fold {k+1}/5 OK')

    oof_ens = 0.5 * oof_lgb + 0.5 * oof_cat

    # ── Evaluar OOF con nueva estrategia ─────────────────────────────────
    correct = det_c = det_t = ml_c = ml_t = 0
    for _, row in train_df.iterrows():
        pid  = row['problem_id']
        opts = {'A':row['option_A'],'B':row['option_B'],'C':row['option_C'],'D':row['option_D']}
        cm   = opts[row['answer']]
        pp   = oof_ens[problem_ids == pid]
        ap, a2, a3, has_bd, _, _, _ = solve_both(row['rules_text'], row['reaction_state'])
        inter = a2 & a3

        if len(ap) == 1 or len(inter) == 1:
            winner = list(ap)[0] if len(ap) == 1 else list(inter)[0]
            det_t += 1
            if winner == cm: det_c += 1; correct += 1
        else:
            cands = (a2 | a3) if (a2 | a3) else set(MECHS)
            ml_t += 1
            sc = {m: pp[i] for i, m in enumerate(MECHS) if m in cands}
            if sc and max(sc, key=sc.get) == cm: ml_c += 1; correct += 1

    print(f'\n── Evaluación OOF ──')
    print(f'Determinístico:  {det_c}/{det_t} = {det_c/det_t:.4f}')
    print(f'ML tiebreak OOF: {ml_c}/{ml_t} = {ml_c/ml_t:.4f}')
    print(f'Combined OOF:    {correct}/{len(train_df)} = {correct/len(train_df):.4f}')

    print('\nEntrenando modelos finales...')
    clf_lgb.fit(X, y)
    clf_cat.fit(X, y)
    print('OK')

Construyendo dataset...
Shape: (28000, 37) | Positivos: 0.250
5-fold CV...
  Fold 1/5 OK
  Fold 2/5 OK
  Fold 3/5 OK
  Fold 4/5 OK
  Fold 5/5 OK

── Evaluación OOF ──
Determinístico:  6110/6343 = 0.9633
ML tiebreak OOF: 492/657 = 0.7489
Combined OOF:    6602/7000 = 0.9431

Entrenando modelos finales...
OK


## 5. Claude API fallback (detección automática)

In [42]:
API_AVAILABLE = None

def check_api_available() -> bool:
    global API_AVAILABLE
    if API_AVAILABLE is not None:
        return API_AVAILABLE
    try:
        payload = json.dumps({
            'model': 'claude-haiku-4-5-20251001',
            'max_tokens': 5,
            'messages': [{'role': 'user', 'content': 'Say A'}]
        }).encode()
        req = urllib.request.Request(
            'https://api.anthropic.com/v1/messages', data=payload,
            headers={'Content-Type': 'application/json'}, method='POST')
        with urllib.request.urlopen(req, timeout=10) as resp:
            data = json.loads(resp.read())
            API_AVAILABLE = 'content' in data
            print('Claude API disponible ✓')
    except Exception as e:
        API_AVAILABLE = False
        print(f'Claude API no disponible ({type(e).__name__}) — continuando sin ella')
    return API_AVAILABLE


def call_claude_api(prompt: str, retries: int = 2):
    for attempt in range(retries):
        try:
            payload = json.dumps({
                'model': 'claude-haiku-4-5-20251001',
                'max_tokens': 10,
                'messages': [{'role': 'user', 'content': prompt}]
            }).encode()
            req = urllib.request.Request(
                'https://api.anthropic.com/v1/messages', data=payload,
                headers={'Content-Type': 'application/json'}, method='POST')
            with urllib.request.urlopen(req, timeout=20) as resp:
                text = json.loads(resp.read())['content'][0]['text'].strip()
                if text and text[0] in 'ABCD': return text[0]
        except Exception:
            if attempt < retries - 1: time.sleep(1)
    return None


def build_api_prompt(rules_text, state_str, options, candidates):
    return (
        'Sos un evaluador lógico preciso. Determiná cuál mecanismo sobrevive aplicando estas reglas.\n\n'
        f'REACTION STATE: {state_str}\n\n'
        f'RULES:\n{rules_text}\n\n'
        f'OPTIONS:\n' + '\n'.join(f'  {k}: {v}' for k,v in options.items()) + '\n\n'
        'SEMÁNTICAS:\n'
        '- 4 mecanismos empiezan activos\n'
        '- blocked gana sobre activated (bloqueado no se reactiva)\n'
        '- override cancela la regla indicada (forward-only)\n'
        '- X dominates Y: si ambos activos, Y se elimina\n'
        '- dominance reversed: invierte X→Y a Y→X; si no hay previa, crea Y→X\n'
        '- exactly N remain, eliminate X: iterativo al final\n\n'
        f'Candidatos: {", ".join(sorted(candidates))}\n'
        'Respondé SOLO con la letra (A, B, C o D).'
    )


check_api_available()

Claude API no disponible (HTTPError) — continuando sin ella


False

## 6. Pipeline de predicción

In [43]:
def predict_one(rules_text, state_str, options):
    ap, a2, a3, has_bd, pm, sv, gf = solve_both(rules_text, state_str)
    inv   = {v: k for k, v in options.items()}
    inter = a2 & a3

    # Determinístico: solver primario da 1
    if len(ap) == 1:
        return inv.get(list(ap)[0], 'A')

    # Determinístico: intersección da 1
    if len(inter) == 1:
        return inv.get(list(inter)[0], 'A')

    # Zero survivors + API
    cands = (a2 | a3) if (a2 | a3) else set(MECHS)
    if len(cands) == 0 and API_AVAILABLE:
        prompt = build_api_prompt(rules_text, state_str, options, set(MECHS))
        resp   = call_claude_api(prompt)
        if resp and resp in options: return resp

    # ML tiebreaker
    if ML_AVAILABLE and clf_lgb is not None:
        na2, na3, nap = len(a2), len(a3), len(ap)
        n_inter = len(inter)
        sc = {}
        for i, m in enumerate(MECHS):
            if m not in cands: continue
            feat = build_feature_vector(
                pm, sv, gf, i,
                1 if m in a2 else 0,
                1 if m in a3 else 0,
                1 if m in ap else 0,
                0, na2, na3, nap,
                1 if m in inter else 0, n_inter,
                1 if has_bd else 0).reshape(1, -1)
            sc[m] = (0.5 * clf_lgb.predict_proba(feat)[0][1] +
                     0.5 * clf_cat.predict_proba(feat)[0][1])
        if sc:
            return inv.get(max(sc, key=sc.get), 'A')

    best = max(cands, key=lambda m: pm[m][2])
    return inv.get(best, 'A')

## 7. Generar submission

In [44]:
print('Generando predicciones para test...')

predictions = []
n_det = n_ml = n_api = n_zero = 0

for i, row in test_df.iterrows():
    opts = {'A':row['option_A'],'B':row['option_B'],'C':row['option_C'],'D':row['option_D']}
    ap, a2, a3, has_bd, _, _, _ = solve_both(row['rules_text'], row['reaction_state'])
    inter = a2 & a3

    if len(ap) == 1 or len(inter) == 1:
        n_det += 1
    elif len(a2 | a3) == 0:
        n_zero += 1
        if API_AVAILABLE: n_api += 1
        else:             n_ml  += 1
    else:
        n_ml += 1

    pred = predict_one(row['rules_text'], row['reaction_state'], opts)
    predictions.append({'problem_id': row['problem_id'], 'answer': pred})

    if (i + 1) % 500 == 0:
        print(f'  {i+1}/{len(test_df)} procesados...')

submission = pd.DataFrame(predictions)
submission.to_csv(SUBMIT_PATH, index=False)

n = len(test_df)
print(f'\n── Estadísticas de inferencia ──')
print(f'Determinístico: {n_det:,} ({n_det/n*100:.1f}%)')
print(f'ML tiebreak:    {n_ml:,} ({n_ml/n*100:.1f}%)')
print(f'API fallback:   {n_api:,} ({n_api/n*100:.1f}%)')
print(f'Zero-survivor:  {n_zero:,} ({n_zero/n*100:.1f}%)')
print(f'\nDistribución de respuestas:')
print(submission["answer"].value_counts().to_string())
print(f'\nSubmission guardado en: {SUBMIT_PATH}')
submission.head(10)

Generando predicciones para test...
  500/3000 procesados...
  1000/3000 procesados...
  1500/3000 procesados...
  2000/3000 procesados...
  2500/3000 procesados...
  3000/3000 procesados...

── Estadísticas de inferencia ──
Determinístico: 2,716 (90.5%)
ML tiebreak:    284 (9.5%)
API fallback:   0 (0.0%)
Zero-survivor:  2 (0.1%)

Distribución de respuestas:
answer
A    760
C    759
B    741
D    740

Submission guardado en: submission_chemical.csv


,problem_id,answer
0,4,A
1,6,B
2,11,D
3,17,A
4,19,C
5,24,C
6,26,C
7,32,C
8,37,B
9,38,D
